#### Salting Technique in Spark
* Salting is used to handle data skew in joins or aggregations by artificially adding a random key to distribute skewed data more evenly across partitions.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, explode, lit, rand, sequence

# Create Spark session
spark = SparkSession.builder.appName("SaltingDemo").getOrCreate()

# Skewed dataset (large number of rows with same key)
df_skewed = spark.createDataFrame(
    [(1, "A"), (1, "B"), (1, "C"), (2, "D"), (3, "E")],
    ["id", "value"]
)

# Small lookup table
df_lookup = spark.createDataFrame(
    [(1, "X"), (2, "Y"), (3, "Z")],
    ["id", "lookup_value"]
)

# SALTING: Add random salt to skewed dataset
salt_count = 3  # number of salts
df_skewed_salted = df_skewed.withColumn("salt", (rand() * salt_count).cast("int"))
df_skewed_salted = df_skewed_salted.withColumn("id_salted", concat_ws("_", col("id"), col("salt")))

# Expand lookup table with same salt range using explode
salt_array = sequence(lit(0), lit(salt_count - 1))  # [0, 1, ..., salt_count-1]
df_lookup_salted = df_lookup.withColumn("salt", explode(salt_array))
df_lookup_salted = df_lookup_salted.withColumn("id_salted", concat_ws("_", col("id"), col("salt")))

# Join on salted keys
df_joined = df_skewed_salted.join(df_lookup_salted, "id_salted").select(df_skewed_salted["id"], "value", "lookup_value")

df_joined.show()
